In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch import nn, optim
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

df = pd.read_csv('allsides_balanced_news_headlines-texts.csv', index_col=0)
df['full_text'] = df['heading'].fillna('') + " " + df['text'].fillna('')

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['bias_rating'])
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
id2label = {i: label for label, i in label2id.items()}


In [ ]:

class TripletBiasDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.groups = df.groupby('title')

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        event = list(self.groups)[idx][1]
        anchor = event.sample(1).iloc[0]
        label = anchor['label']
        positive_pool = event[event['label'] == label]
        positive = positive_pool.sample(1).iloc[0] if len(positive_pool) > 1 else anchor
        negative_pool = event[event['label'] != label]
        negative = negative_pool.sample(1).iloc[0] if not negative_pool.empty else anchor

        def tokenize(text):
            return self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')

        return {
            'anchor': tokenize(anchor['full_text']),
            'positive': tokenize(positive['full_text']),
            'negative': tokenize(negative['full_text']),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:

class HybridBiasModel(nn.Module):
    def __init__(self, model_name='bucketresearch/politicalBiasBERT', num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state.mean(dim=1)
        pooled = self.dropout(pooled)
        return pooled, self.classifier(pooled)


In [ ]:

def train_model(model, dataloader, optimizer, device):
    ce_loss_fn = nn.CrossEntropyLoss()
    contrastive_loss_fn = nn.CosineEmbeddingLoss()
    model.train()
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        def get_embed(part): return model(part['input_ids'].squeeze(1).to(device), part['attention_mask'].squeeze(1).to(device))
        a_emb, a_logits = get_embed(batch['anchor'])
        p_emb, _ = get_embed(batch['positive'])
        n_emb, _ = get_embed(batch['negative'])
        labels = batch['label'].to(device)
        sup_loss = ce_loss_fn(a_logits, labels)
        pos_target = torch.ones(a_emb.size(0)).to(device)
        neg_target = -1 * pos_target
        c_loss = contrastive_loss_fn(a_emb, p_emb, pos_target) + contrastive_loss_fn(a_emb, n_emb, neg_target)
        loss = sup_loss + c_loss
        loss.backward()
        optimizer.step()

def evaluate_model(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            a_emb, a_logits = model(batch['anchor']['input_ids'].squeeze(1).to(device), batch['anchor']['attention_mask'].squeeze(1).to(device))
            preds = torch.argmax(a_logits, dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(batch['label'].numpy())
    print(classification_report(y_true, y_pred, target_names=list(label_encoder.classes_)))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


In [ ]:

def get_embeddings(model, dataloader, device):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            pooled, _ = model(batch['anchor']['input_ids'].squeeze(1).to(device), batch['anchor']['attention_mask'].squeeze(1).to(device))
            embeddings.append(pooled.cpu())
            labels.extend(batch['label'].cpu().numpy())
    return torch.cat(embeddings).numpy(), labels


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bucketresearch/politicalBiasBERT")
model = HybridBiasModel().to(device)

grouped = df.groupby('title')
valid_titles = grouped.filter(lambda x: len(x) == 3 and x['bias_rating'].nunique() == 3)['title'].unique()
df_shared = df[df['title'].isin(valid_titles)].copy()
df_remaining = df[~df.index.isin(df_shared.index)]
min_class_size = df_remaining['bias_rating'].value_counts().min()
balanced_remaining = pd.concat([resample(g, replace=False, n_samples=min_class_size, random_state=42) for _, g in df_remaining.groupby('bias_rating')])
balanced_df = pd.concat([df_shared, balanced_remaining]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, val_df = train_test_split(balanced_df, stratify=balanced_df['label'], test_size=0.2, random_state=42)
train_dataset = TripletBiasDataset(train_df, tokenizer)
val_dataset = TripletBiasDataset(val_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)
for epoch in range(1):
    print(f"Epoch {epoch+1}")
    train_model(model, train_loader, optimizer, device)

print("Evaluating...")
evaluate_model(model, DataLoader(val_dataset, batch_size=16), device)

print("Generating t-SNE...")
X_emb, y_lbl = get_embeddings(model, DataLoader(val_dataset, batch_size=16), device)
X_2d = TSNE(n_components=2, random_state=42).fit_transform(X_emb)
plt.figure(figsize=(10, 6))
for label in set(y_lbl):
    idxs = [i for i, y in enumerate(y_lbl) if y == label]
    plt.scatter(X_2d[idxs, 0], X_2d[idxs, 1], label=id2label[label])
plt.legend()
plt.title("t-SNE Visualization of Bias Embeddings")
plt.show()


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch import nn, optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
import random
import torch.nn.functional as F

df = pd.read_csv('allsides_balanced_news_headlines-texts.csv', index_col=0)
df['full_text'] = df['heading'].fillna('') + " " + df['text'].fillna('')
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['bias_rating'])
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
id2label = {i: label for label, i in label2id.items()}

class TripletBiasDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        anchor = self.df.iloc[idx]
        label = anchor['label']

        positive_pool = self.df[(self.df['label'] == label) & (self.df.index != idx)]
        positive = positive_pool.sample(1).iloc[0] if not positive_pool.empty else anchor

        negative_pool = self.df[self.df['label'] != label]
        negatives = negative_pool.sample(10)

        a_text = anchor['full_text']
        a_input = self.tokenizer(a_text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        a_vec = a_input['input_ids']
        scores = [F.cosine_similarity(a_vec.float(), self.tokenizer(n['full_text'], truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')['input_ids'].float()).mean().item() for _, n in negatives.iterrows()]
        hard_negative = negatives.iloc[np.argmin(scores)]

        def tokenize(text):
            return self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')

        return {
            'anchor': tokenize(anchor['full_text']),
            'positive': tokenize(positive['full_text']),
            'negative': tokenize(hard_negative['full_text']),
            'label': torch.tensor(label, dtype=torch.long)
        }

class HybridBiasModel(nn.Module):
    def __init__(self, model_name='bucketresearch/politicalBiasBERT', num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = True
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state.mean(dim=1)
        pooled = self.dropout(pooled)
        return pooled, self.classifier(pooled)

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.cosine_similarity = nn.CosineSimilarity(dim=-1)

    def forward(self, anchor, positive, negative):
        pos_sim = self.cosine_similarity(anchor, positive) / self.temperature
        neg_sim = self.cosine_similarity(anchor, negative) / self.temperature
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1)
        labels = torch.zeros(anchor.size(0), dtype=torch.long).to(anchor.device)
        return F.cross_entropy(logits, labels)


def train_model(model, dataloader, optimizer, device, loss_fn):
    ce_loss_fn = nn.CrossEntropyLoss()
    model.train()
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        def embed(part):
            return model(part['input_ids'].squeeze(1).to(device), part['attention_mask'].squeeze(1).to(device))
        a_emb, a_logits = embed(batch['anchor'])
        p_emb, _ = embed(batch['positive'])
        n_emb, _ = embed(batch['negative'])
        labels = batch['label'].to(device)
        sup_loss = ce_loss_fn(a_logits, labels)
        c_loss = loss_fn(a_emb, p_emb, n_emb)
        loss = sup_loss + c_loss
        loss.backward()
        optimizer.step()


def evaluate_model(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            a_emb, a_logits = model(batch['anchor']['input_ids'].squeeze(1).to(device), batch['anchor']['attention_mask'].squeeze(1).to(device))
            preds = torch.argmax(a_logits, dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(batch['label'].numpy())
    print(classification_report(y_true, y_pred, target_names=list(label_encoder.classes_)))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bucketresearch/politicalBiasBERT")
model = HybridBiasModel().to(device)

grouped = df.groupby('title')
valid_titles = grouped.filter(lambda x: len(x) == 3 and x['bias_rating'].nunique() == 3)['title'].unique()
df_shared = df[df['title'].isin(valid_titles)].copy()
df_remaining = df[~df.index.isin(df_shared.index)]
min_class_size = df_remaining['bias_rating'].value_counts().min()
balanced_remaining = pd.concat([
    g.sample(min_class_size, random_state=42)
    for _, g in df_remaining.groupby('bias_rating')
])
balanced_df = pd.concat([df_shared, balanced_remaining]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, val_df = train_test_split(balanced_df, stratify=balanced_df['label'], test_size=0.2, random_state=42)
train_dataset = TripletBiasDataset(train_df, tokenizer)
val_dataset = TripletBiasDataset(val_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = NTXentLoss()

for epoch in range(2):
    print(f"Epoch {epoch+1}")
    train_model(model, train_loader, optimizer, device, loss_fn)

In [ ]:
print("Evaluating...")
evaluate_model(model, val_loader, device)